<a href="https://colab.research.google.com/github/adityab-tech/Prism/blob/main/W6_Prism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/shiyu-coder/Kronos.git

fatal: destination path 'Kronos' already exists and is not an empty directory.


In [ ]:
!git clone https://github.com/NeoQuasar/Kronos.git

fatal: destination path 'Kronos' already exists and is not an empty directory.


In [ ]:
%cd Kronos

/content/Kronos


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
sys.path.append("/content/Kronos")
sys.path.append("/content/Kronos/finetune_csv")

In [ ]:
from finetune_csv.config_loader import CustomFinetuneConfig
config = CustomFinetuneConfig(
    "/content/Kronos/finetune_csv/configs/config_ali09988_candle-5min.yaml")

In [ ]:
config.data_path = "/content/drive/MyDrive/PRISM/processed/TCS_processed.csv"
print(config.data_path)

/content/drive/MyDrive/PRISM/processed/TCS_processed.csv


In [ ]:
df = pd.read_csv(config.data_path)
print(df.head())
print(df.columns)

   Unnamed: 0  timestamps         open         high          low        close  \
0          61  2019-04-02  2037.099976  2086.000000  2037.000000  2079.300049   
1          62  2019-04-03  2085.000000  2089.600098  2058.100098  2079.300049   
2          63  2019-04-04  2078.149902  2079.699951  2007.400024  2014.500000   
3          64  2019-04-05  2028.650024  2054.399902  2018.800049  2048.300049   
4          65  2019-04-08  2059.000000  2075.000000  2032.699951  2070.750000   

    volume    return  log_return          MA5         MA10         MA20  \
0  3719663  0.023454    0.023183  2012.360034  2011.235034  2009.327515   
1  2939886  0.000000    0.000000  2031.690039  2016.885034  2013.522516   
2  4397518 -0.031164   -0.031660  2041.010034  2016.055029  2014.842517   
3  3152103  0.016778    0.016639  2050.610034  2019.380029  2017.277521   
4  2194294  0.010960    0.010901  2058.430029  2025.890027  2020.150018   

   Volatility  Volume_Change  Volume_MA20  market_return      

In [ ]:
df = df.rename(columns={
    "Date": "timestamps",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume"
})
df["amount"] = 0
df.to_csv(config.data_path, index=False)

In [ ]:
os.chdir("/content/Kronos/finetune_csv")

In [ ]:
from finetune_base_model import create_dataloaders

In [18]:
print(config.lookback_window)
print(config.predict_window)
print(config.train_ratio)
print(config.val_ratio)
print(config.test_ratio)
print(len(pd.read_csv(config.data_path)))

512
48
0.9
0.1
0.0
1417


In [19]:
config.lookback_window = 30
config.predict_window = 5


In [20]:
(   train_loader,
    val_loader,
    train_dataset,
    val_dataset,
    train_sampler,
    val_sampler
) = create_dataloaders(config)

Creating data loaders...
Original data time range: 2019-04-02 00:00:00 to 2024-12-30 00:00:00
Original data total length: 1417 records
[TRAIN] Training set: first 1275 time points (0.9)
[TRAIN] Training set time range: 2019-04-02 00:00:00 to 2024-06-04 00:00:00
[TRAIN] Data length after split: 1275 records
[TRAIN] Data length: 1275, Available samples: 1240
Original data time range: 2019-04-02 00:00:00 to 2024-12-30 00:00:00
Original data total length: 1417 records
[VAL] Validation set: time points 1276 to 1417 (0.1)
[VAL] Validation set time range: 2024-06-05 00:00:00 to 2024-12-30 00:00:00
[VAL] Data length after split: 142 records
[VAL] Data length: 142, Available samples: 107
Training set size: 1240, Validation set size: 107


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [21]:
print(len(train_dataset))
print(len(val_dataset))

1240
107


In [22]:
batch_x, batch_stamp = next(iter(train_loader))
print(batch_x.shape)
print(batch_stamp.shape)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([32, 36, 6])
torch.Size([32, 36, 5])


In [23]:
from model import Kronos
from model import KronosTokenizer

In [24]:
tokenizer = KronosTokenizer.from_pretrained(
    "NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained(
    "NeoQuasar/Kronos-base")

config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/409M [00:00<?, ?B/s]

In [25]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")

tokenizer.to(device)
model.to(device)
print(device)

cpu


In [26]:
batch_x = batch_x.to(device)

In [27]:
with torch.no_grad():
    (   reconstruction,
        bsq_loss,
        quantized,
        token_ids
    ) = tokenizer(batch_x)

In [28]:
print(type(reconstruction))
print(bsq_loss)
print(quantized.shape)
print(token_ids.shape)

<class 'tuple'>
tensor(-0.0675)
torch.Size([32, 36, 20])
torch.Size([32, 36])


In [29]:
with torch.no_grad():
    token_ids = tokenizer.encode(batch_x)

In [30]:
print(token_ids.shape)
print(token_ids.dtype)

torch.Size([32, 36])
torch.int64


In [31]:
with torch.no_grad():
    prediction = model(token_ids)

TypeError: Kronos.forward() missing 1 required positional argument: 's2_ids'